# Module 4.2 - Conversation Summarisation

**Reference:** [`02-conversation-summarisation.md`](./02-conversation-summarisation.md)

## What you'll do in this notebook

- Ask the LLM to summarise older turns into a short paragraph.
- Build a rolling summary that accumulates as the conversation grows.
- Measure the token saving versus an unbounded history.
- Observe the lossy nature of summarisation by running the account-number recall test.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
encoding = tiktoken.encoding_for_model("gpt-4o-mini")
print(f"OK: client ready, model = {MODEL}")

## Exercise 1 - Summarise a block of turns

In [ ]:
def summarise_turns(turns: list[dict]) -> str:
    """Return a 2-3 sentence summary of the given messages."""
    # TODO:
    # 1. Format the turns as lines of the form 'user: ...' and 'assistant: ...'.
    # 2. Send them to the LLM with a system prompt that asks for a concise
    #    summary that preserves facts, decisions, and open questions.
    #    Use temperature=0.3, max_tokens=150.
    # 3. Return the reply text.
    raise NotImplementedError

example = [
    {"role": "user",      "content": "My account ID is ACC-42."},
    {"role": "assistant", "content": "Thanks. How can I help you today?"},
    {"role": "user",      "content": "I can't log in after changing my password yesterday."},
    {"role": "assistant", "content": "Let's verify a few things. Are you using the latest password on our website or mobile app?"},
    {"role": "user",      "content": "Mobile app. It says invalid credentials."},
]
print(summarise_turns(example))

## Exercise 2 - Rolling-summary chatbot

Keep the last `keep_recent` turns verbatim. When the history gets longer than `2 * keep_recent`, summarise the older half and prepend the summary as a system message.

In [ ]:
class SummarisingChatbot:
    def __init__(self, system_prompt: str, keep_recent: int = 6):
        self.system_prompt = system_prompt
        self.keep_recent = keep_recent
        self.summary = ""
        self.recent: list[dict] = []

    def _maybe_roll_up(self) -> None:
        # TODO: if len(self.recent) > 2 * self.keep_recent:
        #   older = self.recent[:self.keep_recent]
        #   self.recent = self.recent[self.keep_recent:]
        #   new_summary = summarise_turns(older)
        #   self.summary = (self.summary + "\n\n" + new_summary).strip() if self.summary else new_summary
        raise NotImplementedError

    def _build_messages(self, user_message: str) -> list[dict]:
        msgs = [{"role": "system", "content": self.system_prompt}]
        if self.summary:
            msgs.append({"role": "system", "content": f"Summary of earlier conversation:\n{self.summary}"})
        msgs.extend(self.recent)
        msgs.append({"role": "user", "content": user_message})
        return msgs

    def chat(self, user_message: str) -> str:
        messages = self._build_messages(user_message)
        resp = client.chat.completions.create(model=MODEL, messages=messages)
        reply = resp.choices[0].message.content
        self.recent.append({"role": "user", "content": user_message})
        self.recent.append({"role": "assistant", "content": reply})
        self._maybe_roll_up()
        return reply

    def context_tokens(self) -> int:
        parts = [self.system_prompt, self.summary] + [m["content"] for m in self.recent]
        return sum(len(encoding.encode(p)) for p in parts if p)

bot = SummarisingChatbot("You are a helpful support assistant. Keep replies short.", keep_recent=4)
bot.chat("My account ID is ACC-42.")
for i in range(10):
    bot.chat(f"Tell me a short fun fact about topic {i}.")
    print(f"Turn {i+2:2}: summary chars={len(bot.summary):4}, recent={len(bot.recent):2}, context tokens={bot.context_tokens():4}")

## Exercise 3 - The recall test

Ask the bot for the account number it was told at turn 1. A lossy summary might have dropped it. A good summariser should have kept it.

In [ ]:
print("Summary so far:")
print(bot.summary)
print("\nBot's answer to the recall question:")
print(bot.chat("What is my account ID?"))

Try modifying the system prompt in `summarise_turns` to explicitly demand 'preserve all account IDs, order numbers, error codes verbatim'. Does recall improve? This is a preview of selective memory in the next section.

## Bonus - Hierarchical summarisation

For very long conversations, the summary itself grows. Sketch a two-level hierarchy: when `self.summary` exceeds some character budget, summarise *it* into a shorter 'older summary'. Leave the implementation as an exercise - the API is exactly the same as Exercise 1.

## What's next

Summaries forget *by accident*. `03-selective-memory.ipynb` forgets *on purpose* - scoring each turn for importance and dropping the filler first.